# 02 · Transformación Silver – Olist Brazilian E-Commerce
**Origen:** `olist.bronze` (9 tablas Delta, todo como texto)
**Destino:** `olist.silver` → 9 tablas limpias + una tabla `_cuarentena` por entidad + `log_calidad`

Silver es la capa **confiable**: el mismo dato que Bronze, pero:
1. **Tipado**: fechas como `TIMESTAMP`, importes como `DECIMAL(10,2)`, conteos como `INT`. Se usa `try_cast` / `try_to_timestamp`, que devuelven `NULL` en vez de romper el pipeline.
2. **Con nombres de negocio**: en español y sin las erratas del origen (`product_name_lenght` → `largo_nombre`).
3. **Con reglas de calidad**: la fila que no pasa va a **cuarentena** con su motivo. No se borra.
4. **Deduplicado por la clave de negocio** correcta y **cuadrado**: `bronze = sanas + cuarentena`.
5. **Con contrato**: `NOT NULL`, `CHECK` y `PRIMARY KEY` declarados en la tabla misma.

Orden de proceso (por integridad referencial): categorías → productos → clientes → vendedores → pedidos → ítems → pagos → reseñas → geolocalización.

## 1. Parámetros y esquema
El catálogo llega como parámetro, igual que en Bronze, para ejecutar el notebook desde el Workflow.

In [0]:
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql import Window

dbutils.widgets.text("catalogo", "olist")
catalogo = dbutils.widgets.get("catalogo")
bronze = f"{catalogo}.bronze"
silver = f"{catalogo}.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver} COMMENT 'Capa Silver: datos limpios, tipados y con reglas de calidad'")
print("Origen:", bronze, "| Destino:", silver)

## 2. Perfilado: mirar antes de limpiar
Porcentaje de nulos por columna en cada tabla Bronze (solo se listan las columnas que tienen nulos). Este paso decide qué reglas tiene sentido escribir.

In [0]:
TABLAS_BRONZE = ["orders", "order_items", "order_payments", "order_reviews", "customers",
                 "sellers", "products", "geolocation", "category_translation"]

perfil = []
for t in TABLAS_BRONZE:
    df = spark.table(f"{bronze}.{t}")
    columnas = [c for c in df.columns if not c.startswith("_")]
    fila = df.select(F.count("*").alias("__total"),
                     *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in columnas]).first().asDict()
    total = fila.pop("__total")
    for columna, nulos in fila.items():
        if nulos:
            perfil.append((t, columna, total, nulos, round(100 * nulos / total, 2)))

display(spark.createDataFrame(perfil, "tabla string, columna string, filas long, nulos long, pct_nulos double")
        .orderBy(F.desc("pct_nulos")))

**¿Es `review_id` una clave?** Si esta consulta devuelve filas, el mismo `review_id` aparece en más de un pedido. Deduplicar solo por `review_id` borraría reseñas reales (la misma lección que los *remakes* de la actividad de gobierno). Por eso la clave de negocio de reseñas es **`review_id` + `order_id`**.

In [0]:
display(spark.sql(f"""
    SELECT review_id, count(*) AS veces, count(DISTINCT order_id) AS pedidos_distintos
    FROM {bronze}.order_reviews
    GROUP BY review_id
    HAVING count(*) > 1
    ORDER BY veces DESC
    LIMIT 10"""))

## 3. Motor de calidad (se escribe una vez y se usa en cada tabla)
- `evaluar`: agrega `motivos_rechazo`, la lista de reglas que la fila incumple. Una fila sana tiene la lista vacía. Una regla que da `NULL` no rechaza; por eso los nulos se controlan con reglas propias.
- `existe_en`: marca si una clave existe en otra tabla Silver (**integridad referencial**). Así un ítem con un producto inexistente no llega al negocio.
- `procesar`: separa sanas y cuarentena, deduplica por la clave de negocio (o consolida con una agregación), escribe ambas tablas, aplica el contrato, documenta y cuadra las cuentas.

El contrato se quita antes de reescribir y se vuelve a poner después. Así el notebook es **idempotente**: el Workflow puede ejecutarlo todas las veces que haga falta.

In [0]:
def ts(columna):
    """Texto 'yyyy-MM-dd HH:mm:ss' -> TIMESTAMP (NULL si no se puede leer)."""
    return F.expr(f"try_to_timestamp({columna}, 'yyyy-MM-dd HH:mm:ss')")


def num(columna, tipo):
    """Texto -> tipo numérico (NULL si no se puede convertir)."""
    return F.expr(f"try_cast({columna} AS {tipo})")


def texto(columna):
    return F.trim(F.col(columna))


def evaluar(df, reglas):
    """Agrega motivos_rechazo: lista de reglas incumplidas (vacía = fila sana)."""
    motivos = [F.when(F.coalesce(condicion, F.lit(False)), F.lit(motivo))
               for motivo, condicion in reglas.items()]
    return df.withColumn("motivos_rechazo", F.array_compact(F.array(*motivos)))


def existe_en(df, columna, tabla_ref, columna_ref, marca):
    """Left join contra una tabla Silver: marca = True si la clave existe, NULL si no."""
    ref = (spark.table(f"{silver}.{tabla_ref}")
           .select(F.col(columna_ref).alias(columna)).distinct()
           .withColumn(marca, F.lit(True)))
    return df.join(ref, on=columna, how="left")


def quitar_contrato(tabla, checks):
    if not spark.catalog.tableExists(tabla):
        return
    spark.sql(f"ALTER TABLE {tabla} DROP PRIMARY KEY IF EXISTS CASCADE")
    for nombre in checks:
        spark.sql(f"ALTER TABLE {tabla} DROP CONSTRAINT IF EXISTS {nombre}")


def aplicar_contrato(tabla, clave, checks):
    for columna in clave:
        spark.sql(f"ALTER TABLE {tabla} ALTER COLUMN {columna} SET NOT NULL")
    for nombre, expresion in checks.items():
        spark.sql(f"ALTER TABLE {tabla} ADD CONSTRAINT {nombre} CHECK ({expresion})")
    pk = "pk_" + tabla.split(".")[-1]
    spark.sql(f"ALTER TABLE {tabla} ADD CONSTRAINT {pk} PRIMARY KEY ({', '.join(clave)})")


reconciliacion = []


def procesar(nombre, origen, df, reglas, clave, checks=None, orden=None,
             consolidar=None, comentario=""):
    """Cuarentena + deduplicación/consolidación + escritura + contrato + reconciliación."""
    checks = checks or {}
    tabla = f"{silver}.{nombre}"
    ahora = F.current_timestamp()

    evaluado = evaluar(df, reglas)
    evaluado = evaluado.drop(*[c for c in evaluado.columns if c.startswith("_ref_")])

    cuarentena = (evaluado.filter(F.size("motivos_rechazo") > 0)
                  .withColumn("_cuarentena_ts", ahora))
    sanas = evaluado.filter(F.size("motivos_rechazo") == 0).drop("motivos_rechazo")

    if consolidar is not None:
        final = consolidar(sanas)
    else:
        w = Window.partitionBy(*clave).orderBy(*(orden or [F.col("_ingestion_ts").desc()]))
        final = (sanas.withColumn("_fila", F.row_number().over(w))
                      .filter("_fila = 1").drop("_fila"))
    final = final.withColumn("_procesado_ts", ahora)

    quitar_contrato(tabla, checks)
    final.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(tabla)
    (cuarentena.write.mode("overwrite").option("overwriteSchema", True)
               .saveAsTable(f"{tabla}_cuarentena"))
    aplicar_contrato(tabla, clave, checks)

    spark.sql(f"COMMENT ON TABLE {tabla} IS '{comentario}'")
    spark.sql(f"COMMENT ON TABLE {tabla}_cuarentena IS "
              f"'Filas de bronze.{origen} que no superaron las reglas de calidad de silver.{nombre}'")
    spark.sql(f"ALTER TABLE {tabla} SET TAGS ('capa' = 'silver', 'dominio' = 'ecommerce')")

    n_bronze = spark.table(f"{bronze}.{origen}").count()
    n_sanas = sanas.count()
    n_silver = spark.table(tabla).count()
    n_cuarentena = spark.table(f"{tabla}_cuarentena").count()
    fila = (nombre, n_bronze, n_silver, n_cuarentena, n_sanas - n_silver,
            n_bronze == n_sanas + n_cuarentena)
    reconciliacion.append(fila)
    print(f"{tabla}: bronze {n_bronze:,} -> silver {n_silver:,} | "
          f"cuarentena {n_cuarentena:,} | consolidadas {n_sanas - n_silver:,}")
    return fila

## 4. Categorías (`category_translation`)
Tabla de referencia portugués → inglés. Clave: `categoria_pt`.

In [0]:
df = spark.table(f"{bronze}.category_translation").select(
    texto("product_category_name").alias("categoria_pt"),
    texto("product_category_name_english").alias("categoria_en"),
    "_ingestion_ts",
)

procesar(
    "categorias", "category_translation", df,
    reglas={
        "categoria_pt nula": F.col("categoria_pt").isNull(),
        "categoria_en nula": F.col("categoria_en").isNull(),
    },
    clave=["categoria_pt"],
    comentario="Traducción de categorías de producto, portugués a inglés. Una fila por categoría",
)

## 5. Productos (`products`)
- Se corrigen las erratas del origen (`lenght` → `largo`).
- Se agrega `categoria_en` desde `silver.categorias`. Si una categoría no tiene traducción, se conserva el nombre en portugués. Si el producto no tiene categoría, queda `NULL` y en Gold se mostrará como `sin_categoria`.
- Un producto sin categoría **no** va a cuarentena: sigue siendo un producto que se vende.

In [0]:
categorias = spark.table(f"{silver}.categorias").select("categoria_pt", "categoria_en")

df = (spark.table(f"{bronze}.products")
      .select(
          texto("product_id").alias("producto_id"),
          texto("product_category_name").alias("categoria_pt"),
          num("product_name_lenght", "INT").alias("largo_nombre"),
          num("product_description_lenght", "INT").alias("largo_descripcion"),
          num("product_photos_qty", "INT").alias("cant_fotos"),
          num("product_weight_g", "INT").alias("peso_g"),
          num("product_length_cm", "INT").alias("largo_cm"),
          num("product_height_cm", "INT").alias("alto_cm"),
          num("product_width_cm", "INT").alias("ancho_cm"),
          "_ingestion_ts")
      .join(categorias, "categoria_pt", "left")
      .withColumn("categoria_en", F.coalesce("categoria_en", "categoria_pt")))

procesar(
    "productos", "products", df,
    reglas={
        "producto_id nulo": F.col("producto_id").isNull(),
        "medidas negativas": (F.col("peso_g") < 0) | (F.col("largo_cm") < 0)
                             | (F.col("alto_cm") < 0) | (F.col("ancho_cm") < 0),
    },
    clave=["producto_id"],
    checks={"peso_no_negativo": "peso_g IS NULL OR peso_g >= 0"},
    comentario="Catálogo de productos con categoría traducida. Una fila por producto",
)

## 6. Clientes (`customers`)
Olist genera un `customer_id` **por pedido**; la persona real es `customer_unique_id`. Se conservan los dos: `cliente_id` para unir con pedidos y `cliente_unico_id` para contar clientes reales en Gold.

El prefijo postal llega como texto. `lpad` asegura 5 dígitos aunque se hayan perdido ceros a la izquierda.

In [0]:
df = spark.table(f"{bronze}.customers").select(
    texto("customer_id").alias("cliente_id"),
    texto("customer_unique_id").alias("cliente_unico_id"),
    F.lpad(texto("customer_zip_code_prefix"), 5, "0").alias("cp_prefijo"),
    F.lower(texto("customer_city")).alias("ciudad"),
    F.upper(texto("customer_state")).alias("estado"),
    "_ingestion_ts",
)

procesar(
    "clientes", "customers", df,
    reglas={
        "cliente_id nulo": F.col("cliente_id").isNull(),
        "cliente_unico_id nulo": F.col("cliente_unico_id").isNull(),
        "estado nulo": F.col("estado").isNull(),
        "estado inválido": ~F.col("estado").rlike("^[A-Z]{2}$"),
    },
    clave=["cliente_id"],
    checks={"estado_dos_letras": "length(estado) = 2"},
    comentario="Clientes por pedido. cliente_unico_id identifica a la persona. Una fila por cliente_id",
)

## 7. Vendedores (`sellers`)

In [0]:
df = spark.table(f"{bronze}.sellers").select(
    texto("seller_id").alias("vendedor_id"),
    F.lpad(texto("seller_zip_code_prefix"), 5, "0").alias("cp_prefijo"),
    F.lower(texto("seller_city")).alias("ciudad"),
    F.upper(texto("seller_state")).alias("estado"),
    "_ingestion_ts",
)

procesar(
    "vendedores", "sellers", df,
    reglas={
        "vendedor_id nulo": F.col("vendedor_id").isNull(),
        "estado nulo": F.col("estado").isNull(),
        "estado inválido": ~F.col("estado").rlike("^[A-Z]{2}$"),
    },
    clave=["vendedor_id"],
    checks={"estado_dos_letras": "length(estado) = 2"},
    comentario="Vendedores del marketplace. Una fila por vendedor",
)

## 8. Pedidos (`orders`)
Reglas de rechazo (el pedido no sirve para el negocio):
- sin `pedido_id`, sin fecha de compra legible o con un estado fuera del dominio conocido;
- con un cliente que no existe en `silver.clientes`;
- entregado **antes** de comprarse (fecha incoherente).

Un pedido `delivered` sin fecha de entrega **no** se rechaza: es una venta real. Solo queda sin `dias_entrega` en Gold.

In [0]:
ESTADOS_PEDIDO = ["created", "approved", "invoiced", "processing",
                  "shipped", "delivered", "canceled", "unavailable"]

df = spark.table(f"{bronze}.orders").select(
    texto("order_id").alias("pedido_id"),
    texto("customer_id").alias("cliente_id"),
    F.lower(texto("order_status")).alias("estado_pedido"),
    ts("order_purchase_timestamp").alias("fecha_compra"),
    ts("order_approved_at").alias("fecha_aprobacion"),
    ts("order_delivered_carrier_date").alias("fecha_envio"),
    ts("order_delivered_customer_date").alias("fecha_entrega"),
    ts("order_estimated_delivery_date").alias("fecha_entrega_estimada"),
    "_ingestion_ts",
)
df = existe_en(df, "cliente_id", "clientes", "cliente_id", "_ref_cliente")

lista_estados = ", ".join(f"'{e}'" for e in ESTADOS_PEDIDO)
procesar(
    "pedidos", "orders", df,
    reglas={
        "pedido_id nulo": F.col("pedido_id").isNull(),
        "fecha de compra ilegible": F.col("fecha_compra").isNull(),
        "estado desconocido": F.col("estado_pedido").isNull() | ~F.col("estado_pedido").isin(ESTADOS_PEDIDO),
        "cliente inexistente": F.col("_ref_cliente").isNull(),
        "entrega antes de la compra": F.col("fecha_entrega") < F.col("fecha_compra"),
    },
    clave=["pedido_id"],
    checks={
        "estado_pedido_valido": f"estado_pedido IN ({lista_estados})",
        "entrega_posterior_compra": "fecha_entrega IS NULL OR fecha_entrega >= fecha_compra",
    },
    comentario="Pedidos con fechas tipadas y estado validado. Una fila por pedido",
)

## 9. Ítems del pedido (`order_items`)
Es la tabla de hechos de la venta: un pedido puede tener varios ítems. Clave: `pedido_id` + `item_num`.
Se valida que pedido, producto y vendedor existan en Silver, y que el precio sea positivo.

In [0]:
df = spark.table(f"{bronze}.order_items").select(
    texto("order_id").alias("pedido_id"),
    num("order_item_id", "INT").alias("item_num"),
    texto("product_id").alias("producto_id"),
    texto("seller_id").alias("vendedor_id"),
    ts("shipping_limit_date").alias("fecha_limite_envio"),
    num("price", "DECIMAL(10,2)").alias("precio"),
    num("freight_value", "DECIMAL(10,2)").alias("flete"),
    "_ingestion_ts",
)
df = existe_en(df, "pedido_id", "pedidos", "pedido_id", "_ref_pedido")
df = existe_en(df, "producto_id", "productos", "producto_id", "_ref_producto")
df = existe_en(df, "vendedor_id", "vendedores", "vendedor_id", "_ref_vendedor")

procesar(
    "items_pedido", "order_items", df,
    reglas={
        "clave incompleta": F.col("pedido_id").isNull() | F.col("item_num").isNull(),
        "precio no positivo": F.col("precio").isNull() | (F.col("precio") <= 0),
        "flete negativo o ilegible": F.col("flete").isNull() | (F.col("flete") < 0),
        "pedido inexistente": F.col("_ref_pedido").isNull(),
        "producto inexistente": F.col("_ref_producto").isNull(),
        "vendedor inexistente": F.col("_ref_vendedor").isNull(),
    },
    clave=["pedido_id", "item_num"],
    checks={"precio_positivo": "precio > 0", "flete_no_negativo": "flete >= 0"},
    comentario="Items vendidos por pedido, precio y flete en BRL. Una fila por pedido e item",
)

## 10. Pagos (`order_payments`)
Un pedido puede pagarse en varias partes (`secuencia`). Si existen pagos con tipo `not_defined`, van a cuarentena: no se sabe cómo se cobró.

In [0]:
df = spark.table(f"{bronze}.order_payments").select(
    texto("order_id").alias("pedido_id"),
    num("payment_sequential", "INT").alias("secuencia"),
    F.lower(texto("payment_type")).alias("tipo_pago"),
    num("payment_installments", "INT").alias("cuotas"),
    num("payment_value", "DECIMAL(10,2)").alias("monto"),
    "_ingestion_ts",
)
df = existe_en(df, "pedido_id", "pedidos", "pedido_id", "_ref_pedido")

procesar(
    "pagos", "order_payments", df,
    reglas={
        "clave incompleta": F.col("pedido_id").isNull() | F.col("secuencia").isNull(),
        "tipo de pago no definido": F.col("tipo_pago").isNull() | (F.col("tipo_pago") == "not_defined"),
        "monto negativo o ilegible": F.col("monto").isNull() | (F.col("monto") < 0),
        "cuotas negativas": F.col("cuotas") < 0,
        "pedido inexistente": F.col("_ref_pedido").isNull(),
    },
    clave=["pedido_id", "secuencia"],
    checks={"monto_no_negativo": "monto >= 0", "cuotas_no_negativas": "cuotas IS NULL OR cuotas >= 0"},
    comentario="Pagos por pedido en BRL. Una fila por pedido y secuencia de pago",
)

## 11. Reseñas (`order_reviews`)
Clave de negocio **`resena_id` + `pedido_id`** (ver el perfilado). Si hubiera repetidos exactos, se queda la respuesta más reciente.

In [0]:
df = spark.table(f"{bronze}.order_reviews").select(
    texto("review_id").alias("resena_id"),
    texto("order_id").alias("pedido_id"),
    num("review_score", "INT").alias("puntaje"),
    texto("review_comment_title").alias("titulo"),
    texto("review_comment_message").alias("comentario"),
    ts("review_creation_date").alias("fecha_creacion"),
    ts("review_answer_timestamp").alias("fecha_respuesta"),
    "_ingestion_ts",
)
df = existe_en(df, "pedido_id", "pedidos", "pedido_id", "_ref_pedido")

procesar(
    "resenas", "order_reviews", df,
    reglas={
        "clave incompleta": F.col("resena_id").isNull() | F.col("pedido_id").isNull(),
        "puntaje fuera de rango": F.col("puntaje").isNull() | ~F.col("puntaje").between(1, 5),
        "pedido inexistente": F.col("_ref_pedido").isNull(),
    },
    clave=["resena_id", "pedido_id"],
    orden=[F.col("fecha_respuesta").desc_nulls_last(), F.col("_ingestion_ts").desc()],
    checks={"puntaje_1_a_5": "puntaje BETWEEN 1 AND 5"},
    comentario="Reseñas de clientes, puntaje de 1 a 5. Una fila por reseña y pedido",
)

## 12. Geolocalización (`geolocation`)
El origen trae **muchos puntos por cada prefijo postal**. Silver los **consolida** en una fila por `cp_prefijo` (coordenada promedio, ciudad y estado más frecuentes), para que Gold pueda unir sin duplicar filas.

Antes de consolidar, los puntos fuera del rectángulo de Brasil van a cuarentena: un promedio con un punto en otro continente movería la coordenada.

In [0]:
def consolidar_geo(df):
    return (df.groupBy("cp_prefijo")
              .agg(F.round(F.avg("latitud"), 6).alias("latitud"),
                   F.round(F.avg("longitud"), 6).alias("longitud"),
                   F.mode("ciudad").alias("ciudad"),
                   F.mode("estado").alias("estado"),
                   F.count("*").alias("puntos_origen"),
                   F.max("_ingestion_ts").alias("_ingestion_ts")))


df = spark.table(f"{bronze}.geolocation").select(
    F.lpad(texto("geolocation_zip_code_prefix"), 5, "0").alias("cp_prefijo"),
    num("geolocation_lat", "DOUBLE").alias("latitud"),
    num("geolocation_lng", "DOUBLE").alias("longitud"),
    F.lower(texto("geolocation_city")).alias("ciudad"),
    F.upper(texto("geolocation_state")).alias("estado"),
    "_ingestion_ts",
)

procesar(
    "geolocalizacion", "geolocation", df,
    reglas={
        "cp_prefijo nulo": F.col("cp_prefijo").isNull(),
        "coordenadas ilegibles": F.col("latitud").isNull() | F.col("longitud").isNull(),
        "fuera de Brasil": ~F.col("latitud").between(-34, 5.5) | ~F.col("longitud").between(-74, -34.5),
    },
    clave=["cp_prefijo"],
    consolidar=consolidar_geo,
    checks={"latitud_en_brasil": "latitud BETWEEN -34 AND 5.5",
            "longitud_en_brasil": "longitud BETWEEN -74 AND -34.5"},
    comentario="Coordenada promedio por prefijo postal, solo puntos dentro de Brasil. Una fila por cp_prefijo",
)

## 13. Reconciliación y registro de calidad
Todo lo que entró está en algún sitio: `filas_bronze = filas sanas + filas_cuarentena`. Las sanas, a su vez, son `filas_silver + filas_consolidadas` (duplicados eliminados o puntos agrupados).

Si alguna tabla no cuadra, el notebook **falla a propósito**: el Workflow se detiene y Gold no se construye con datos incompletos.

In [0]:
log = (spark.createDataFrame(
           reconciliacion,
           "tabla string, filas_bronze long, filas_silver long, filas_cuarentena long, "
           "filas_consolidadas long, cuadra boolean")
       .withColumn("fecha_proceso", F.current_timestamp()))

log.write.mode("append").saveAsTable(f"{silver}.log_calidad")
spark.sql(f"COMMENT ON TABLE {silver}.log_calidad IS 'Historial de reconciliación bronze vs silver por ejecución'")
display(log)

no_cuadran = [r[0] for r in reconciliacion if not r[5]]
if no_cuadran:
    raise Exception(f"Reconciliación fallida en: {no_cuadran}. Revisar joins y tipado antes de pasar a Gold.")
print("Reconciliación OK: bronze = sanas + cuarentena en todas las tablas")

### 13.1 · ¿Por qué se rechazaron filas?
Recuento por tabla y motivo. Estas filas no se pierden: alguien del negocio decide si se corrigen en el origen o se descartan.

In [0]:
entidades = sorted({r[0] for r in reconciliacion})
motivos = reduce(lambda a, b: a.unionByName(b), [
    spark.table(f"{silver}.{e}_cuarentena")
         .select(F.lit(e).alias("tabla"), F.explode("motivos_rechazo").alias("motivo"))
    for e in entidades
])
display(motivos.groupBy("tabla", "motivo").count().orderBy(F.desc("count")))

## 14. El contrato funciona
Intentamos meter un ítem con precio negativo por fuera del pipeline. **Debe fallar**, y el error nombra la restricción `precio_positivo`.

In [0]:
mala = (spark.table(f"{silver}.items_pedido").limit(1)
        .withColumn("precio", F.lit(-1).cast("decimal(10,2)")))

try:
    mala.write.mode("append").saveAsTable(f"{silver}.items_pedido")
    print("Se insertó. Esto NO debería pasar.")
except Exception as e:
    print("La tabla rechazó la fila, como debe:")
    print(str(e)[:300])

## 15. Gobierno: documentar y etiquetar
- **Comentarios de columna**: los lee quien no estuvo en el proyecto (y Genie).
- **Etiqueta `sensibilidad = personal`** en los datos que identifican o ubican a una persona. Sirve para encontrarlos y aplicarles políticas.

In [0]:
COMENTARIOS = {
    "pedidos": {
        "pedido_id": "Identificador del pedido. Clave de negocio",
        "estado_pedido": "Estado del pedido en el flujo de Olist: delivered, shipped, canceled, etc.",
        "fecha_entrega_estimada": "Fecha de entrega prometida al cliente al momento de la compra",
    },
    "clientes": {
        "cliente_id": "Id del cliente para un pedido. Olist genera uno nuevo por cada pedido",
        "cliente_unico_id": "Id de la persona. Usar este para contar clientes reales",
    },
    "items_pedido": {
        "precio": "Precio del item en reales (BRL), sin flete",
        "flete": "Costo de envío del item en reales (BRL)",
    },
    "resenas": {
        "puntaje": "Satisfacción del cliente, de 1 a 5",
    },
}
for tabla, columnas in COMENTARIOS.items():
    for columna, comentario in columnas.items():
        spark.sql(f"ALTER TABLE {silver}.{tabla} ALTER COLUMN {columna} COMMENT '{comentario}'")

DATOS_PERSONALES = {
    "clientes": ["cliente_unico_id", "cp_prefijo", "ciudad"],
    "vendedores": ["cp_prefijo"],
}
for tabla, columnas in DATOS_PERSONALES.items():
    for columna in columnas:
        spark.sql(f"ALTER TABLE {silver}.{tabla} ALTER COLUMN {columna} SET TAGS ('sensibilidad' = 'personal')")

print("Columnas documentadas y etiquetadas")

## 16. Inventario e historial

In [0]:
display(spark.sql(f"""
    SELECT table_name AS tabla, comment AS descripcion
    FROM {catalogo}.information_schema.tables
    WHERE table_schema = 'silver'
    ORDER BY table_name"""))

display(spark.sql(f"DESCRIBE HISTORY {silver}.pedidos").select("version", "timestamp", "operation"))